In [35]:
import pandas as pd
import re
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report

In [36]:
raw = pd.read_csv(
    "../data/raw/Amazon_Reviews.csv",
    engine="python",
    on_bad_lines="skip"
)

raw.columns = raw.columns.str.strip()

print(raw.shape)
print(raw.columns)

raw.head()

(21214, 9)
Index(['Reviewer Name', 'Profile Link', 'Country', 'Review Count',
       'Review Date', 'Rating', 'Review Title', 'Review Text',
       'Date of Experience'],
      dtype='str')


,Reviewer Name,Profile Link,Country,Review Count,Review Date,Rating,Review Title,Review Text,Date of Experience
0,Eugene ath,/users/66e8185ff1598352d6b3701a,US,1 review,2024-09-16T13:44:26.000Z,Rated 1 out of 5 stars,A Store That Doesn't Want to Sell Anything,"I registered on the website, tried to order a ...","September 16, 2024"
1,Daniel ohalloran,/users/5d75e460200c1f6a6373648c,GB,9 reviews,2024-09-16T18:26:46.000Z,Rated 1 out of 5 stars,Had multiple orders one turned up and…,Had multiple orders one turned up and driver h...,"September 16, 2024"
2,p fisher,/users/546cfcf1000064000197b88f,GB,90 reviews,2024-09-16T21:47:39.000Z,Rated 1 out of 5 stars,I informed these reprobates,I informed these reprobates that I WOULD NOT B...,"September 16, 2024"
3,Greg Dunn,/users/62c35cdbacc0ea0012ccaffa,AU,5 reviews,2024-09-17T07:15:49.000Z,Rated 1 out of 5 stars,Advertise one price then increase it on website,I have bought from Amazon before and no proble...,"September 17, 2024"
4,Sheila Hannah,/users/5ddbe429478d88251550610e,GB,8 reviews,2024-09-16T18:37:17.000Z,Rated 1 out of 5 stars,If I could give a lower rate I would,If I could give a lower rate I would! I cancel...,"September 16, 2024"


In [37]:
TEXT_COL = "Review Text"
TITLE_COL = "Review Title"
RATING_COL = "Rating"

sentiment_df = raw[[TITLE_COL, TEXT_COL, RATING_COL]].copy()

sentiment_df = sentiment_df.dropna(subset=[TEXT_COL, RATING_COL]).copy()

sentiment_df[TITLE_COL] = sentiment_df[TITLE_COL].fillna("")
sentiment_df[TEXT_COL] = sentiment_df[TEXT_COL].fillna("")

sentiment_df["combined_text"] = (
    sentiment_df[TITLE_COL].astype(str) + " " +
    sentiment_df[TEXT_COL].astype(str)
)

sentiment_df.head()

,Review Title,Review Text,Rating,combined_text
0,A Store That Doesn't Want to Sell Anything,"I registered on the website, tried to order a ...",Rated 1 out of 5 stars,A Store That Doesn't Want to Sell Anything I r...
1,Had multiple orders one turned up and…,Had multiple orders one turned up and driver h...,Rated 1 out of 5 stars,Had multiple orders one turned up and… Had mul...
2,I informed these reprobates,I informed these reprobates that I WOULD NOT B...,Rated 1 out of 5 stars,I informed these reprobates I informed these r...
3,Advertise one price then increase it on website,I have bought from Amazon before and no proble...,Rated 1 out of 5 stars,Advertise one price then increase it on websit...
4,If I could give a lower rate I would,If I could give a lower rate I would! I cancel...,Rated 1 out of 5 stars,If I could give a lower rate I would If I coul...


In [39]:
def rating_to_sentiment(rating):
    rating_text = str(rating).lower()

    match = re.search(r"(\d+)", rating_text)

    if not match:
        return None

    stars = int(match.group(1))

    if stars <= 2:
        return "negative"
    elif stars == 3:
        return "neutral"
    elif stars >= 4:
        return "positive"
    else:
        return None

In [40]:
sentiment_df["sentiment"] = sentiment_df[RATING_COL].apply(rating_to_sentiment)

sentiment_df = sentiment_df.dropna(subset=["sentiment"]).copy()

sentiment_df["sentiment"].value_counts()

sentiment
negative    14350
positive     5820
neutral       885
Name: count, dtype: int64

In [41]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [42]:
sentiment_df["clean_text"] = sentiment_df["combined_text"].apply(clean_text)

sentiment_df = sentiment_df[
    sentiment_df["clean_text"].str.len() > 0
].copy()

sentiment_df = sentiment_df[
    sentiment_df["clean_text"] != "review text not found"
].copy()

sentiment_df["language"] = "en"

sentiment_clean = sentiment_df[["clean_text", "sentiment", "language"]].copy()

sentiment_clean.head()

,clean_text,sentiment,language
0,a store that doesn t want to sell anything i r...,negative,en
1,had multiple orders one turned up and had mult...,negative,en
2,i informed these reprobates i informed these r...,negative,en
3,advertise one price then increase it on websit...,negative,en
4,if i could give a lower rate i would if i coul...,negative,en


In [43]:
print("Total rows:", sentiment_clean.shape[0])
print("Unique reviews:", sentiment_clean["clean_text"].nunique())

print("\nSentiment distribution:")
print(sentiment_clean["sentiment"].value_counts())

print("\nSentiment percentage:")
print(sentiment_clean["sentiment"].value_counts(normalize=True) * 100)

Total rows: 21055
Unique reviews: 20995

Sentiment distribution:
sentiment
negative    14350
positive     5820
neutral       885
Name: count, dtype: int64

Sentiment percentage:
sentiment
negative    68.154833
positive    27.641890
neutral      4.203277
Name: proportion, dtype: float64


In [44]:
sentiment_clean.to_csv(
    "../data/processed/sentiment_clean.csv",
    index=False
)

In [45]:
sentiment_train, sentiment_test = train_test_split(
    sentiment_clean,
    test_size=0.2,
    random_state=42,
    stratify=sentiment_clean["sentiment"]
)

print("Train distribution:")
print(sentiment_train["sentiment"].value_counts())

print("\nTest distribution:")
print(sentiment_test["sentiment"].value_counts())

Train distribution:
sentiment
negative    11480
positive     4656
neutral       708
Name: count, dtype: int64

Test distribution:
sentiment
negative    2870
positive    1164
neutral      177
Name: count, dtype: int64


In [46]:
sentiment_train.to_csv("../data/processed/sentiment_train.csv", index=False)
sentiment_test.to_csv("../data/processed/sentiment_test.csv", index=False)

In [47]:
negative_train = sentiment_train[sentiment_train["sentiment"] == "negative"]
positive_train = sentiment_train[sentiment_train["sentiment"] == "positive"]
neutral_train = sentiment_train[sentiment_train["sentiment"] == "neutral"]

neutral_upsampled = neutral_train.sample(
    n=len(positive_train),
    replace=True,
    random_state=42
)

sentiment_train_balanced = pd.concat([
    negative_train,
    positive_train,
    neutral_upsampled
]).sample(frac=1, random_state=42).reset_index(drop=True)

print("Balanced training distribution:")
print(sentiment_train_balanced["sentiment"].value_counts())

Balanced training distribution:
sentiment
negative    11480
neutral      4656
positive     4656
Name: count, dtype: int64


In [48]:
X_train = sentiment_train_balanced["clean_text"]
y_train = sentiment_train_balanced["sentiment"]

X_test = sentiment_test["clean_text"]
y_test = sentiment_test["sentiment"]

### Logistic Regression

In [49]:
sentiment_lr_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=70000,
        ngram_range=(1, 3),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True
    )),
    ("classifier", LogisticRegression(
        max_iter=5000,
        class_weight="balanced",
        solver="lbfgs",
        C=1.0
    ))
])

sentiment_lr_model.fit(X_train, y_train)

y_pred_lr = sentiment_lr_model.predict(X_test)

print("Logistic Regression Classification Report")
print(classification_report(y_test, y_pred_lr, zero_division=0))

Logistic Regression Classification Report
              precision    recall  f1-score   support

    negative       0.95      0.95      0.95      2870
     neutral       0.28      0.23      0.25       177
    positive       0.89      0.90      0.90      1164

    accuracy                           0.91      4211
   macro avg       0.70      0.69      0.70      4211
weighted avg       0.90      0.91      0.90      4211



### LinearSVC

In [50]:
sentiment_svc_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=70000,
        ngram_range=(1, 3),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True
    )),
    ("classifier", LinearSVC(
        class_weight="balanced",
        max_iter=7000,
        C=1.0
    ))
])

sentiment_svc_model.fit(X_train, y_train)

y_pred_svc = sentiment_svc_model.predict(X_test)

print("LinearSVC Classification Report")
print(classification_report(y_test, y_pred_svc, zero_division=0))

LinearSVC Classification Report
              precision    recall  f1-score   support

    negative       0.94      0.97      0.96      2870
     neutral       0.34      0.11      0.16       177
    positive       0.89      0.91      0.90      1164

    accuracy                           0.92      4211
   macro avg       0.72      0.66      0.67      4211
weighted avg       0.90      0.92      0.91      4211



### Compare Models


In [51]:
lr_report = classification_report(
    y_test,
    y_pred_lr,
    output_dict=True,
    zero_division=0
)

svc_report = classification_report(
    y_test,
    y_pred_svc,
    output_dict=True,
    zero_division=0
)

sentiment_model_comparison = pd.DataFrame({
    "model": [
        "Logistic Regression",
        "LinearSVC"
    ],
    "accuracy": [
        lr_report["accuracy"],
        svc_report["accuracy"]
    ],
    "macro_f1": [
        lr_report["macro avg"]["f1-score"],
        svc_report["macro avg"]["f1-score"]
    ],
    "weighted_f1": [
        lr_report["weighted avg"]["f1-score"],
        svc_report["weighted avg"]["f1-score"]
    ],
    "neutral_f1": [
        lr_report["neutral"]["f1-score"],
        svc_report["neutral"]["f1-score"]
    ]
})

sentiment_model_comparison = sentiment_model_comparison.sort_values(
    by="macro_f1",
    ascending=False
).reset_index(drop=True)

display(sentiment_model_comparison)

sentiment_model_comparison.to_csv(
    "../reports/sentiment_model_comparison.csv",
    index=False
)

,model,accuracy,macro_f1,weighted_f1,neutral_f1
0,Logistic Regression,0.907148,0.698824,0.904911,0.251534
1,LinearSVC,0.919259,0.673510,0.908001,0.163090


In [52]:
best_model_name = sentiment_model_comparison.loc[0, "model"]

if best_model_name == "Logistic Regression":
    final_sentiment_model = sentiment_lr_model
    y_pred_best = y_pred_lr
else:
    final_sentiment_model = sentiment_svc_model
    y_pred_best = y_pred_svc

print("Best sentiment model:", best_model_name)
print("Best macro F1:", sentiment_model_comparison.loc[0, "macro_f1"])

Best sentiment model: Logistic Regression
Best macro F1: 0.6988240580389117


In [53]:
joblib.dump(
    final_sentiment_model,
    "../models/sentiment_model.pkl"
)

['../models/sentiment_model.pkl']

### Saving Wrong Predictions

In [54]:
wrong_sentiment_predictions = pd.DataFrame({
    "text": X_test.reset_index(drop=True),
    "actual_sentiment": y_test.reset_index(drop=True),
    "predicted_sentiment": pd.Series(y_pred_best)
})

wrong_sentiment_predictions = wrong_sentiment_predictions[
    wrong_sentiment_predictions["actual_sentiment"] !=
    wrong_sentiment_predictions["predicted_sentiment"]
]

wrong_sentiment_predictions.to_csv(
    "../reports/wrong_sentiment_predictions.csv",
    index=False
)

print("Wrong predictions:", wrong_sentiment_predictions.shape[0])

display(wrong_sentiment_predictions.head(20))

Wrong predictions: 391


,text,actual_sentiment,predicted_sentiment
13,worst site on the internet for refunds worst w...,negative,positive
14,i used to love amazon now i hate them i used t...,negative,positive
22,no problems yet have been a prime no problems ...,positive,neutral
41,top of the pack need to show others how it s d...,positive,negative
58,amazon turns out to be great my wife persuaded...,positive,negative
69,amazon changing delivery times lately when i h...,negative,neutral
71,the blame isn t on amazon because the the blam...,positive,negative
76,damages brand new car is insured is not damage...,negative,neutral
85,they amazon warehouse deals sent me a beat up ...,negative,positive
97,what s going on with amazon deliveries lately ...,neutral,negative


### German Fallback Rules

In [55]:
german_negative_words = [
    "schlecht", "defekt", "fehler", "problem", "probleme",
    "nicht funktioniert", "kaputt", "enttäuscht", "unzufrieden",
    "beschwerde", "verzögerung", "ausfall", "störung",
    "dringend", "kritisch", "inakzeptabel", "frustriert",
    "fehlgeschlagen"
]

german_positive_words = [
    "gut", "sehr gut", "zufrieden", "danke", "hilfreich",
    "gelöst", "funktioniert", "schnell", "perfekt",
    "ausgezeichnet", "erfolgreich"
]

def german_rule_sentiment(text):
    text = str(text).lower()

    negative_score = sum(word in text for word in german_negative_words)
    positive_score = sum(word in text for word in german_positive_words)

    if negative_score > positive_score:
        return "negative"
    elif positive_score > negative_score:
        return "positive"
    else:
        return "neutral"

In [56]:
def predict_ticket_sentiment(text, language):
    language = str(language).lower()

    if language in ["de", "german", "deutsch"]:
        return german_rule_sentiment(text)
    else:
        return final_sentiment_model.predict([str(text)])[0]

### Applying Sentiment to Ticket Dataset

In [57]:
tickets = pd.read_csv("../data/processed/test.csv")

print(tickets.shape)
print(tickets.columns)

tickets.head()

(5718, 5)
Index(['clean_text', 'type', 'queue', 'priority', 'language'], dtype='str')


,clean_text,type,queue,priority,language
0,potential data breach in healthcare system the...,Incident,Human Resources,low,en
1,hilfe zur entwicklung digitaler wachstumsmetho...,Incident,Returns and Exchanges,high,de
2,issue encountered assistance needed,Problem,Technical Support,high,en
3,benötigte unterstützung bei datenintegration e...,Incident,General Inquiry,low,de
4,anzeigeproblem für projekt-milestones das feat...,Incident,Technical Support,low,de


In [59]:
tickets["predicted_sentiment"] = tickets.apply(
    lambda row: predict_ticket_sentiment(
        row["clean_text"],
        row["language"]
    ),
    axis=1
)

tickets[["clean_text", "language", "predicted_sentiment"]].head(20)

,clean_text,language,predicted_sentiment
0,potential data breach in healthcare system the...,en,negative
1,hilfe zur entwicklung digitaler wachstumsmetho...,de,negative
2,issue encountered assistance needed,en,neutral
3,benötigte unterstützung bei datenintegration e...,de,negative
4,anzeigeproblem für projekt-milestones das feat...,de,negative
5,detailed information on billing options for sm...,en,positive
6,enhanced security implement advanced security ...,de,neutral
7,problem mit datensicherheit ich möchte sie dar...,de,negative
8,gerät zur bildschirmaufnahme läuft unregelmäßi...,de,positive
9,enhanced security for medical data management ...,en,negative


In [60]:
tickets[
    ["clean_text", "type", "queue", "priority", "language", "predicted_sentiment"]
].to_csv(
    "../reports/ticket_sentiment_predictions.csv",
    index=False
)